# GPT-2 stats analysis

This notebook computes:
- Embedding covariance and eigen spectrum
- Effective rank (99% energy) for Q, K, V, and out projection weights
- Hidden state covariance and eigen spectrum after parameter-free LayerNorm
- Mean log eigenvalue per layer and plots

Results are saved under results/<model>.

## 1. Imports and dependencies

This cell imports PyTorch, Transformers, Datasets, plotting, and small utilities used in the rest of the notebook.

In [1]:
import csv
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
import requests
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


/user/luotianze/zjk/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

Adjust model names, dataset, sequence length, batch size, number of batches, and output directory here. Use `embedding_sample_size` to speed up embedding covariance if needed.

In [ ]:
@dataclass
class Config:
    model_names = ["Qwen/Qwen3-4B", "Qwen/Qwen3-8B"]
    """
    models to choose from:
    gpt2
    gpt2-large
    Qwen/Qwen3-0.6B
    Qwen/Qwen3-1.7B
    Qwen/Qwen3-4B
    Qwen/Qwen3-8B
    """
    dataset_name = "Salesforce/wikitext"
    dataset_config = "wikitext-2-raw-v1"
    dataset_mirror: Optional[str] = "https://hf-mirror.com"
    model_mirror_base: Optional[str] = "https://hf-mirror.com"
    max_length = 128
    batch_size = 8
    num_batches = 8
    embedding_sample_size = None  # set an int to speed up embedding covariance
    cov_dtype = torch.float32
    eps = 1e-6
    output_dir = Path("results")


cfg = Config()
cfg.output_dir.mkdir(exist_ok=True)


## 3. Helper functions

Defines the parameter-free LayerNorm, covariance accumulation, eigenvalue sorting/plotting, effective rank computation, and batch construction.

In [3]:
def layernorm_no_params(x: torch.Tensor, eps: float) -> torch.Tensor:
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    return (x - mean) / torch.sqrt(var + eps)


def maybe_sample_rows(x: torch.Tensor, max_rows: Optional[int], seed: int = 42) -> torch.Tensor:
    if not max_rows or x.shape[0] <= max_rows:
        return x
    gen = torch.Generator().manual_seed(seed)
    idx = torch.randperm(x.shape[0], generator=gen)[:max_rows]
    return x[idx]


def covariance_from_rows(x: torch.Tensor) -> torch.Tensor:
    x = x.to(torch.float32)
    x = x - x.mean(dim=0, keepdim=True)
    return (x.T @ x) / (x.shape[0] - 1)


class CovAccumulator:
    def __init__(self, dim: int, dtype: torch.dtype):
        self.dim = dim
        self.dtype = dtype
        self.count = 0
        self.sum = torch.zeros(dim, dtype=dtype)
        self.sum_xtx = torch.zeros(dim, dim, dtype=dtype)

    def update(self, x: torch.Tensor) -> None:
        x = x.to(self.dtype)
        self.sum += x.sum(dim=0)
        self.sum_xtx += x.T @ x
        self.count += x.shape[0]

    def covariance(self) -> torch.Tensor:
        if self.count < 2:
            raise ValueError("Not enough samples for covariance")
        mean = self.sum / self.count
        cov = (self.sum_xtx - self.count * torch.outer(mean, mean)) / (self.count - 1)
        return cov


def eigvals_sorted(cov: torch.Tensor, eps: float) -> torch.Tensor:
    eigvals = torch.linalg.eigvalsh(cov.double())
    eigvals = torch.clamp(eigvals, min=eps)
    return torch.flip(eigvals, dims=[0])


def _piecewise_log2_x(x, linear_max: int = 16) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    y = x.copy()
    mask = x > linear_max
    if np.any(mask):
        step = linear_max - 1
        y[mask] = linear_max + step * np.log2(x[mask] / linear_max)
    return y


def _set_piecewise_log2_xaxis(ax, max_x: int, linear_max: int = 16) -> None:
    ticks = [1]
    if linear_max <= max_x:
        ticks.append(linear_max)
    value = linear_max * 2
    while value <= max_x:
        ticks.append(value)
        value *= 2
    if ticks[-1] != max_x:
        ticks.append(max_x)
    ticks = [t for t in sorted(set(ticks)) if 1 <= t <= max_x]

    tick_positions = _piecewise_log2_x(ticks, linear_max=linear_max)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([str(int(t)) for t in ticks])
    ax.set_xlim(
        _piecewise_log2_x([1], linear_max=linear_max)[0],
        _piecewise_log2_x([max_x], linear_max=linear_max)[0],
    )


def plot_loglog_eigs(eigvals: torch.Tensor, out_path: Path, title: str) -> None:
    n = eigvals.shape[0]
    xs = np.arange(1, n + 1)
    ys = eigvals.cpu().numpy()
    plt.figure(figsize=(6, 4))
    ax = plt.gca()
    ax.plot(_piecewise_log2_x(xs), ys, linewidth=1.0)
    ax.set_yscale("log")
    _set_piecewise_log2_xaxis(ax, n)
    ax.set_xlabel("Principal component index")
    ax.set_ylabel("Eigenvalue")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()


def effective_rank_99(weight: torch.Tensor) -> tuple[int, float]:
    s = torch.linalg.svdvals(weight.float())
    energy = torch.cumsum(s * s, dim=0)
    total = energy[-1]
    k = int(torch.searchsorted(energy, 0.99 * total).item() + 1)
    denom = min(weight.shape)
    return k, k / denom


def build_batches(tokenizer, texts, batch_size: int, max_length: int, num_batches: int):
    batches = []
    needed = batch_size * num_batches
    texts = texts[:needed]
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        if len(batch_texts) < batch_size:
            break
        enc = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=max_length,
        )
        batches.append(enc)
    return batches


## 4. Load data and build batches

Loads Wikitext-2, filters empty lines, tokenizes, and prepares a fixed number of batches for repeatable statistics.

In [4]:
#try:
#    dataset = load_dataset(cfg.dataset_name, cfg.dataset_config, split="train")
#    texts = [t for t in dataset["text"] if t.strip()]
#    print(f"Loaded {len(texts)} texts")
#except Exception as e:
#   print(f"Primary dataset download failed: {e}")
if 1:
    if cfg.dataset_mirror:
        print(f"Attempting to download dataset from mirror: {cfg.dataset_mirror}")
        resp = requests.get(cfg.dataset_mirror, timeout=30)
        resp.raise_for_status()
        content = resp.text
        lines = [ln for ln in content.splitlines() if ln.strip()]
        texts = []
        for line in lines:
            try:
                import json
                obj = json.loads(line)
                if isinstance(obj, dict) and "text" in obj:
                    texts.append(obj["text"])
                else:
                    texts.append(line)
            except Exception:
                texts.append(line)
        texts = [t for t in texts if t.strip()]
        print(f"Loaded {len(texts)} texts from mirror")
    else:
        raise

Attempting to download dataset from mirror: https://hf-mirror.com


Loaded 227 texts from mirror


## 5. Per-model analysis

Runs all requested statistics for one model: embedding covariance/eigen spectrum, QKV/out projection effective rank, hidden-state covariance after parameter-free LayerNorm, per-layer eigen plots, and log-eigenvalue means.

In [5]:
def _get_blocks(model):
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "model") and hasattr(model.model, "decoder") and hasattr(model.model.decoder, "layers"):
        return model.model.decoder.layers
    raise AttributeError("Unsupported model block layout")


def _get_hidden_size(config) -> int:
    return getattr(config, "n_embd", None) or getattr(config, "hidden_size", None)


def _get_attn_weights(block, hidden_size: int):
    attn = getattr(block, "attn", None) or getattr(block, "self_attn", None)
    if attn is None:
        raise AttributeError("Block has no attention module")

    if hasattr(attn, "c_attn") and hasattr(attn, "c_proj"):
        w_qkv = attn.c_attn.weight.detach().cpu()
        w_q = w_qkv[:, :hidden_size]
        w_k = w_qkv[:, hidden_size : 2 * hidden_size]
        w_v = w_qkv[:, 2 * hidden_size :]
        w_o = attn.c_proj.weight.detach().cpu()
        return w_q, w_k, w_v, w_o

    q_proj = getattr(attn, "q_proj", None)
    k_proj = getattr(attn, "k_proj", None)
    v_proj = getattr(attn, "v_proj", None)
    o_proj = getattr(attn, "o_proj", None)
    if all([q_proj, k_proj, v_proj, o_proj]):
        return (
            q_proj.weight.detach().cpu(),
            k_proj.weight.detach().cpu(),
            v_proj.weight.detach().cpu(),
            o_proj.weight.detach().cpu(),
        )

    raise AttributeError("Unsupported attention layout")

In [6]:
def analyze_embedding_stats(
    model,
    model_name: str,
    cfg: Config,
    model_dir: Path
) -> None:
    wte_full = model.get_input_embeddings().weight.detach().cpu()
    wte = maybe_sample_rows(wte_full, cfg.embedding_sample_size)
    cov_emb = covariance_from_rows(wte)
    eig_emb = eigvals_sorted(cov_emb, cfg.eps)
    plot_loglog_eigs(eig_emb, model_dir / "embedding_eigs.png", f"{model_name} embedding eigs")
    emb_logmean = float(torch.log(eig_emb).mean().item())

    with open(model_dir / "embedding_stats.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["log_eig_mean"])
        writer.writerow([emb_logmean])

    wte_f = wte_full.float()
    norms = torch.linalg.norm(wte_f, dim=1, keepdim=True)
    norms = torch.clamp(norms, min=cfg.eps)
    wte_unit = wte_f / norms
    vocab_size = wte_unit.shape[0]
    sample_size = min(128, vocab_size)
    if vocab_size < 2:
        nn_mean_cos = float("nan")
        mean_pairwise_cos = float("nan")
    else:
        gen = torch.Generator().manual_seed(42)
        sample_idx = torch.randperm(vocab_size, generator=gen)[:sample_size]
        sample = wte_unit[sample_idx]
        sims = sample @ wte_unit.T
        sims[torch.arange(sample_size), sample_idx] = -1.0
        nn_mean_cos = float(sims.max(dim=1).values.mean().item())
        # Mean off-diagonal cosine similarity via sum of unit vectors.
        sum_vec = wte_unit.double().sum(dim=0)
        sum_sq = float((sum_vec @ sum_vec).item())
        mean_pairwise_cos = (sum_sq - vocab_size) / (vocab_size * (vocab_size - 1))

    with open(model_dir / "embedding_similarity_stats.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["sample_size", "nn_mean_cosine", "mean_pairwise_cosine"])
        writer.writerow([sample_size, nn_mean_cos, mean_pairwise_cos])

In [7]:
def analyze_effective_ranks(
    blocks,
    hidden_size: int,
    model_name: str,
    model_dir: Path
) -> None:
    ranks = {"q": [], "k": [], "v": [], "o": []}
    for block in tqdm(blocks, desc=f"SVD ranks ({model_name})"):
        w_q, w_k, w_v, w_o = _get_attn_weights(block, hidden_size)

        _, rq = effective_rank_99(w_q)
        _, rk = effective_rank_99(w_k)
        _, rv = effective_rank_99(w_v)
        _, ro = effective_rank_99(w_o)

        ranks["q"].append(rq)
        ranks["k"].append(rk)
        ranks["v"].append(rv)
        ranks["o"].append(ro)

    with open(model_dir / "effective_ranks.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["layer", "q", "k", "v", "o"])
        for i in range(len(ranks["q"])):
            writer.writerow([i, ranks["q"][i], ranks["k"][i], ranks["v"][i], ranks["o"][i]])

    plt.figure(figsize=(7, 4))
    layers = list(range(len(ranks["q"])))
    plt.plot(layers, ranks["q"], label="Q")
    plt.plot(layers, ranks["k"] , label="K")
    plt.plot(layers, ranks["v"], label="V")
    plt.plot(layers, ranks["o"], label="Out")
    plt.xlabel("Layer")
    plt.ylabel("Normalized effective rank (99%)")
    plt.title(f"{model_name} effective rank vs depth")
    plt.legend()
    plt.tight_layout()
    plt.savefig(model_dir / "effective_rank_vs_depth.png")
    plt.close()

In [8]:
def analyze_hidden_state_spectra(
    model,
    batches,
    hidden_size: int,
    num_layers: int,
    device: torch.device,
    cfg: Config,
    model_name: str,
    model_dir: Path,
) -> None:
    accs = [CovAccumulator(hidden_size, cfg.cov_dtype) for _ in range(num_layers)]

    for batch in tqdm(batches, desc=f"Hidden states ({model_name})"):
        batch = {k: v.to(device) for k, v in batch.items()}
        attn_mask = batch.get("attention_mask")
        token_mask = attn_mask.reshape(-1).bool() if attn_mask is not None else None
        with torch.no_grad():
            outputs = model(**batch, output_hidden_states=True, use_cache=False)
        hidden_states = outputs.hidden_states[1:]

        for i, h in enumerate(hidden_states):
            x = h.reshape(-1, h.shape[-1])
            if token_mask is not None:
                x = x[token_mask]
                if x.numel() == 0:
                    continue
            x = layernorm_no_params(x, cfg.eps)
            accs[i].update(x.cpu())

    logmeans = []
    eigs_by_layer = {}
    layer_step = 3 if num_layers <= 24 else 5
    selected_layers = list(range(0, num_layers, layer_step))
    if (num_layers - 1) not in selected_layers:
        selected_layers.append(num_layers - 1)

    for i, acc in tqdm(list(enumerate(accs)), desc=f"Eig spectra ({model_name})"):
        cov = acc.covariance()
        eigs = eigvals_sorted(cov, cfg.eps)
        plot_loglog_eigs(
            eigs,
            model_dir / f"layer_{i:02d}_eigs.png",
            f"{model_name} layer {i} eigs",
        )
        if i in selected_layers:
            eigs_by_layer[i] = eigs
        logmeans.append(float(torch.log(eigs).mean().item()))

    plt.figure(figsize=(7, 4))
    max_n = None
    for i in selected_layers:
        eigs = eigs_by_layer.get(i)
        if eigs is None:
            continue
        n = eigs.shape[0]
        max_n = n
        xs = np.arange(1, n + 1)
        ys = eigs.cpu().numpy()
        plt.plot(_piecewise_log2_x(xs), ys, linewidth=1.0, label=f"Layer {i}")
    ax = plt.gca()
    ax.set_yscale("log")
    if max_n is not None:
        _set_piecewise_log2_xaxis(ax, max_n)
    ax.set_xlabel("Principal component index")
    ax.set_ylabel("Eigenvalue")
    ax.set_title(f"{model_name} eigs by layer (every {layer_step})")
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(model_dir / "layer_eigs_overview.png")
    plt.close()

    with open(model_dir / "layer_log_eig_mean.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["layer", "log_eig_mean"])
        for i, v in enumerate(logmeans):
            writer.writerow([i, v])

    plt.figure(figsize=(7, 4))
    plt.plot(list(range(len(logmeans))), logmeans, marker="o", linewidth=1.0)
    plt.xlabel("Layer")
    plt.ylabel("Mean log eigenvalue")
    plt.title(f"{model_name} log-eig mean vs depth")
    plt.tight_layout()
    plt.savefig(model_dir / "log_eig_mean_vs_depth.png")
    plt.close()


In [9]:
def analyze_model(model_name: str, texts, cfg: Config, device: torch.device) -> None:
    def _download_model_from_mirror(mirror_base: str, model_name: str) -> Path:
        print(f"Downloading model from mirror: {mirror_base} for {model_name}")
        local_dir = snapshot_download(
            repo_id=model_name,
            repo_type="model",
            endpoint=mirror_base,
        )
        return Path(local_dir)

    print(f"Loading model: {model_name}")
#    try:
#        tokenizer = AutoTokenizer.from_pretrained(model_name)
#    except Exception as e:
#        print(f\
    if 1:
        if cfg.model_mirror_base:
            local_dir = _download_model_from_mirror(cfg.model_mirror_base, model_name)
            tokenizer = AutoTokenizer.from_pretrained(str(local_dir))
        else:
            raise

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    batches = build_batches(tokenizer, texts, cfg.batch_size, cfg.max_length, cfg.num_batches)
    print(f"Prepared {len(batches)} batches for {model_name}")

#    try:
#        model = AutoModelForCausalLM.from_pretrained(model_name)
#    except Exception as e:
#        print(f\
    if 1:
        if cfg.model_mirror_base:
            local_dir = _download_model_from_mirror(cfg.model_mirror_base, model_name)
            model = AutoModelForCausalLM.from_pretrained(str(local_dir))
        else:
            raise

    model.to(device)
    model.eval()

    model_dir = cfg.output_dir / model_name.replace("/", "_")
    model_dir.mkdir(parents=True, exist_ok=True)

    analyze_embedding_stats(model, model_name, cfg, model_dir)

    blocks = _get_blocks(model)
    hidden_size = _get_hidden_size(model.config)
    if hidden_size is None:
        raise AttributeError("Model config missing hidden size")

    analyze_effective_ranks(blocks, hidden_size, model_name, model_dir)
    analyze_hidden_state_spectra(
        model,
        batches,
        hidden_size,
        len(blocks),
        device,
        cfg,
        model_name,
        model_dir,
    )

## 6. Run the pipeline

Selects CPU/GPU and executes the analysis for the models chosen.

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

for model_name in cfg.model_names:
    analyze_model(model_name, texts, cfg, device)

print("Done. See results/<model> for outputs.")

Device: cpu
Loading model: Qwen/Qwen3-4B


/user/luotianze/zjk/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 141625.85it/s]


Prepared 1 batches for Qwen/Qwen3-4B


Eig spectra (Qwen/Qwen3-4B): 100%|██████████| 36/36 [00:57<00:00,  1.59s/it]


Loading model: Qwen/Qwen3-8B


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 168220.75it/s]


Prepared 1 batches for Qwen/Qwen3-8B


Eig spectra (Qwen/Qwen3-8B): 100%|██████████| 36/36 [02:24<00:00,  4.01s/it]


Done. See results/<model> for outputs.
